# 🔬 PathOS Distilled — Evaluation Notebook

> This notebook evaluates format compliance, clinical accuracy, anti-hedging behavior, chain-of-thought quality, and response calibration across three question types (YN / Tissue / Open). A **head-to-head comparison section** is included if you have both models available.

## 1. Setup & Installation

In [ ]:
!pip install llama-cpp-python huggingface_hub pandas scikit-learn matplotlib seaborn tqdm --quiet

import os, re, json, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Dependencies loaded.')

## 2. Load Model (Q8_0)

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID  = "dhairyapandya/pathos-gemma4-distilled-GGUF"
FILENAME = "pathos-Q8_0.gguf"

print(f"⬇️  Downloading {FILENAME} (~4.95 GB) from {REPO_ID} ...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"   Cached at: {model_path}")

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=os.cpu_count(),
    verbose=False
)
print('✅ Model loaded (Q8_0, 8-bit).')

## 3. System Prompt & Inference Helper

The same 6 inference-time techniques are baked into the weights via SFT distillation — the system prompt reinforces them.

In [ ]:
SYSTEM_PROMPT = """You are an expert AI pathologist.
Given a histopathological description or question, provide a precise, evidence-based answer.
Always wrap your final answer in <answer>...</answer> tags.
Never say 'cannot determine' — reason from available evidence to the most probable conclusion.
Use chain-of-thought reasoning before your answer when appropriate."""

HEDGE_PHRASES = [
    "cannot determine", "unable to determine", "not possible to determine",
    "insufficient information", "cannot be determined", "more information needed",
    "it is not possible", "i cannot say"
]

def run_inference(question: str, max_tokens: int = 600, temperature: float = 0.1) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]
    t0 = time.time()
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )
    elapsed   = time.time() - t0
    raw       = resp["choices"][0]["message"]["content"]
    ans_match = re.search(r'<answer>(.*?)</answer>', raw, re.DOTALL | re.IGNORECASE)
    extracted = ans_match.group(1).strip() if ans_match else None
    hedged    = any(p in raw.lower() for p in HEDGE_PHRASES)

    # CoT signal: did the model reason before answering?
    pre_answer = raw[:ans_match.start()].strip() if ans_match else raw
    has_cot    = len(pre_answer.split()) > 30   # heuristic: >30 words of reasoning

    return {
        "raw":             raw,
        "answer":          extracted,
        "has_answer_tag":  ans_match is not None,
        "hedged":          hedged,
        "has_cot":         has_cot,
        "cot_word_count":  len(pre_answer.split()),
        "latency_s":       round(elapsed, 2),
        "tokens":          resp["usage"]["completion_tokens"]
    }

# ── Smoke test ────────────────────────────────────────────────────────────────
result = run_inference("Is malignancy present in a tissue showing irregular nuclear borders and a mitotic rate of 9/10 HPF?")
print("Raw output:\n", result['raw'])
print("\n--- Parsed ---")
for k, v in result.items():
    if k != 'raw':
        print(f"  {k}: {v}")

## 4. Evaluation Dataset

Three question types mirroring the model's per-type templates:

| Type | n | Scoring |
|---|---|---|
| **YN** | 8 | Exact yes/no extraction |
| **Tissue** | 7 | Keyword match against expected terms |
| **Open** | 6 | Keyword coverage score |

The Open set includes one **grading** and one **mechanism** question specifically probing CoT routing.

In [ ]:
EVAL_DATASET = [
    # ── YES / NO ──────────────────────────────────────────────────────────────
    {
        "id": "yn_001", "type": "YN",
        "question": "A biopsy shows cells with irregular nuclear contours, prominent nucleoli, and a mitotic rate of 8/10 HPF. Is malignancy present?",
        "expected": "yes", "label": 1
    },
    {
        "id": "yn_002", "type": "YN",
        "question": "Skin biopsy reveals uniform keratinocytes with intact rete ridges and no atypia. Is malignancy present?",
        "expected": "no", "label": 0
    },
    {
        "id": "yn_003", "type": "YN",
        "question": "Breast tissue shows pleomorphic ductal cells with comedonecrosis and a cribriform architecture within ducts. Is this DCIS?",
        "expected": "yes", "label": 1
    },
    {
        "id": "yn_004", "type": "YN",
        "question": "A lymph node shows reactive follicular hyperplasia with preserved architecture and no Reed-Sternberg cells. Is lymphoma present?",
        "expected": "no", "label": 0
    },
    {
        "id": "yn_005", "type": "YN",
        "question": "Colon biopsy shows glandular structures infiltrating through the muscularis mucosa into the submucosa. Is submucosal invasion present?",
        "expected": "yes", "label": 1
    },
    {
        "id": "yn_006", "type": "YN",
        "question": "Thyroid FNA shows follicular cells with nuclear grooves, intranuclear pseudo-inclusions, and ground-glass nuclei. Is papillary carcinoma suspected?",
        "expected": "yes", "label": 1
    },
    {
        "id": "yn_007", "type": "YN",
        "question": "A prostate biopsy shows Gleason pattern 3 glands only, no perineural invasion, Gleason score 3+3=6. Is this high-grade prostate cancer?",
        "expected": "no", "label": 0
    },
    {
        "id": "yn_008", "type": "YN",
        "question": "Lung biopsy shows squamous differentiation with keratin pearls and intercellular bridges. Is squamous cell carcinoma present?",
        "expected": "yes", "label": 1
    },

    # ── TISSUE / CLASSIFICATION ────────────────────────────────────────────────
    {
        "id": "tissue_001", "type": "Tissue",
        "question": "Biopsy shows nests of cells with clear cytoplasm, prominent cell membranes, and a delicate vascular network. What is the most likely diagnosis?",
        "expected": "renal cell carcinoma",
        "keywords": ["renal cell", "clear cell", "kidney carcinoma"]
    },
    {
        "id": "tissue_002", "type": "Tissue",
        "question": "Histology shows small blue round cells in sheets with Homer-Wright rosettes and a high N/C ratio in a child. What tumor is this?",
        "expected": "neuroblastoma",
        "keywords": ["neuroblastoma", "PNET", "small round blue cell"]
    },
    {
        "id": "tissue_003", "type": "Tissue",
        "question": "Skin biopsy shows melanocytes at the dermoepidermal junction with pagetoid upward spread, mitoses, and nuclear pleomorphism. What is the diagnosis?",
        "expected": "melanoma",
        "keywords": ["melanoma", "malignant melanoma"]
    },
    {
        "id": "tissue_004", "type": "Tissue",
        "question": "Gastric biopsy shows discohesive signet ring cells within mucosa, minimal gland formation, and prominent desmoplastic stroma. What carcinoma subtype?",
        "expected": "signet ring cell carcinoma",
        "keywords": ["signet ring", "poorly cohesive", "diffuse gastric"]
    },
    {
        "id": "tissue_005", "type": "Tissue",
        "question": "Bone marrow biopsy shows 90% blasts expressing CD34, TdT, CD19, and CD10. What is the diagnosis?",
        "expected": "B-cell acute lymphoblastic leukemia",
        "keywords": ["ALL", "acute lymphoblastic", "B-ALL", "leukemia", "precursor B"]
    },
    {
        "id": "tissue_006", "type": "Tissue",
        "question": "Lung biopsy shows glandular cells growing along alveolar walls (lepidic pattern) without stromal, pleural, or vascular invasion. What is the classification?",
        "expected": "adenocarcinoma in situ",
        "keywords": ["adenocarcinoma in situ", "AIS", "lepidic", "non-invasive"]
    },
    {
        "id": "tissue_007", "type": "Tissue",
        "question": "Liver biopsy shows zone 3 hepatocyte necrosis, Mallory-Denk bodies, neutrophilic infiltrate, and ballooning degeneration. What is the diagnosis?",
        "expected": "alcoholic hepatitis",
        "keywords": ["alcoholic hepatitis", "steatohepatitis", "ASH"]
    },

    # ── OPEN / REASONING ──────────────────────────────────────────────────────
    {
        "id": "open_001", "type": "Open",
        "question": "Explain the histological difference between Gleason grade 3 and Gleason grade 4 prostate cancer.",
        "keywords": ["discrete", "well-formed", "fused", "cribriform", "infiltrative", "poorly formed"]
    },
    {
        "id": "open_002", "type": "Open",
        "question": "What IHC markers differentiate pleural mesothelioma from lung adenocarcinoma?",
        "keywords": ["calretinin", "WT1", "D2-40", "CEA", "TTF-1", "MOC-31", "BerEP4", "CK5/6"]
    },
    {
        "id": "open_003", "type": "Open",
        "question": "Describe the histological features distinguishing follicular adenoma from follicular carcinoma of the thyroid.",
        "keywords": ["capsular invasion", "vascular invasion", "transcapsular", "angioinvasion", "full thickness"]
    },
    {
        "id": "open_004", "type": "Open",
        "question": "What microscopic features distinguish Crohn's disease from ulcerative colitis?",
        "keywords": ["transmural", "granuloma", "skip lesions", "fissuring", "full thickness", "continuous"]
    },
    {
        "id": "open_005", "type": "Open",
        "question": "A 55-year-old patient has a lung mass. Biopsy shows large cells with abundant cytoplasm and prominent nucleoli but no gland formation and no keratinization. What is the differential and how would you proceed with IHC?",
        "keywords": ["large cell", "carcinoma", "TTF-1", "p40", "napsin", "IHC", "NUT", "exclusion"]
    },
    {
        "id": "open_006", "type": "Open",
        "question": "Explain the mechanism by which KRAS mutation drives colorectal carcinogenesis and its histological correlates.",
        "keywords": ["KRAS", "RAS", "MAP kinase", "EGFR", "proliferation", "serrated", "adenoma", "microsatellite"]
    }
]

df_meta = pd.DataFrame(EVAL_DATASET)
print(f"Total eval cases: {len(df_meta)}")
print(df_meta.groupby('type').size().rename('count').to_string())

## 5. Run Evaluations

In [ ]:
results = []
for case in tqdm(EVAL_DATASET, desc="Evaluating (Q8_0)"):
    out = run_inference(case["question"])
    results.append({**case, **out})

rdf = pd.DataFrame(results)
print(f"✅ Evaluation complete — {len(rdf)} cases.")

## 6. Scoring

In [ ]:
def score_yn(row):
    text = (row["answer"] or row["raw"]).lower()
    has_yes = bool(re.search(r'\byes\b', text))
    has_no  = bool(re.search(r'\bno\b',  text))
    if row["expected"] == "yes":
        return 1 if has_yes else 0
    return 1 if has_no and not has_yes else 0

def score_tissue(row):
    text = (row["answer"] or row["raw"]).lower()
    kws  = row.get("keywords", [row.get("expected", "")])
    return 1 if any(k.lower() in text for k in kws) else 0

def score_open(row):
    text = (row["answer"] or row["raw"]).lower()
    kws  = row.get("keywords", [])
    if not kws:
        return None
    matched = sum(1 for k in kws if k.lower() in text)
    return round(matched / len(kws), 3)

def score_row(row):
    if row["type"] == "YN":     return score_yn(row)
    if row["type"] == "Tissue": return score_tissue(row)
    if row["type"] == "Open":   return score_open(row)

rdf["score"] = rdf.apply(score_row, axis=1)

print(rdf[["id", "type", "has_answer_tag", "has_cot", "hedged", "score", "latency_s", "tokens"]].to_string(index=False))

## 7. Aggregate Metrics

In [ ]:
yn_rows     = rdf[rdf["type"] == "YN"]
tissue_rows = rdf[rdf["type"] == "Tissue"]
open_rows   = rdf[rdf["type"] == "Open"]

metrics = {
    "Model":                         "pathos-gemma4-distilled-GGUF (Q8_0)",
    "Training":                      "SFT distillation",
    "Total cases":                   len(rdf),
    "Format Compliance (<answer>)": f"{rdf['has_answer_tag'].mean():.1%}",
    "Anti-Hedging Rate":            f"{(~rdf['hedged']).mean():.1%}",
    "CoT Presence Rate":            f"{rdf['has_cot'].mean():.1%}",
    "Avg CoT Word Count":           f"{rdf['cot_word_count'].mean():.0f} words",
    "YN Accuracy":                  f"{yn_rows['score'].mean():.1%}  (n={len(yn_rows)})",
    "Tissue Classification Acc":    f"{tissue_rows['score'].mean():.1%}  (n={len(tissue_rows)})",
    "Open QA Keyword Coverage":     f"{open_rows['score'].mean():.1%}  (n={len(open_rows)})",
    "Avg Latency (s)":              f"{rdf['latency_s'].mean():.2f}",
    "Avg Output Tokens":            f"{rdf['tokens'].mean():.0f}",
    "Tokens / second":              f"{(rdf['tokens'] / rdf['latency_s']).mean():.1f}",
}

print("\n" + "═"*58)
print("  PathOS Distilled (Q8_0) — Evaluation Summary")
print("═"*58)
for k, v in metrics.items():
    print(f"  {k:<40} {v}")
print("═"*58)

## 8. Visualisations

In [ ]:
fig = plt.figure(figsize=(18, 13))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.42)

PALETTE = ['#4C78A8', '#F28E2B', '#59A14F', '#E15759', '#76B7B2', '#B07AA1']

# ── 1. Score per question type ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ts = rdf.groupby("type")["score"].mean()
bars = ax1.bar(ts.index, ts.values, color=PALETTE[:3], edgecolor='white', linewidth=1.5)
ax1.set_ylim(0, 1.15)
ax1.set_title('Score by Question Type', fontweight='bold')
ax1.set_ylabel('Score')
for bar, val in zip(bars, ts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.0%}', ha='center', fontsize=11)

# ── 2. Behaviour metrics ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bm_labels = ['Format\nCompliance', 'Anti-Hedging\nRate', 'CoT\nPresence']
bm_vals   = [
    rdf['has_answer_tag'].mean(),
    (~rdf['hedged']).mean(),
    rdf['has_cot'].mean()
]
bars2 = ax2.bar(bm_labels, bm_vals, color=PALETTE[3:6], edgecolor='white', linewidth=1.5)
ax2.set_ylim(0, 1.15)
ax2.set_title('Model Behaviour Metrics', fontweight='bold')
ax2.set_ylabel('Rate')
for bar, val in zip(bars2, bm_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.0%}', ha='center', fontsize=11)

# ── 3. CoT word count by question type ────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
cot_data = [rdf[rdf['type'] == t]['cot_word_count'].values for t in ['YN', 'Tissue', 'Open']]
bp = ax3.boxplot(cot_data, labels=['YN', 'Tissue', 'Open'], patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE[:3]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax3.set_title('CoT Length by Question Type', fontweight='bold')
ax3.set_ylabel('Word Count (pre-answer reasoning)')

# ── 4. YN confusion matrix ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
yn_df = rdf[rdf['type'] == 'YN'].copy()
def pred_yn(row):
    text = (row['answer'] or row['raw']).lower()
    hy = bool(re.search(r'\byes\b', text))
    hn = bool(re.search(r'\bno\b',  text))
    if hy and not hn: return 1
    if hn and not hy: return 0
    return -1
yn_df['predicted'] = yn_df.apply(pred_yn, axis=1)
yn_valid = yn_df[yn_df['predicted'] != -1]
if len(yn_valid) >= 2:
    cm = confusion_matrix(yn_valid['label'], yn_valid['predicted'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax4, cmap='Blues',
                xticklabels=['Pred: No','Pred: Yes'],
                yticklabels=['True: No','True: Yes'])
    ax4.set_title('YN Confusion Matrix', fontweight='bold')
else:
    ax4.text(0.5, 0.5, 'Insufficient YN data', ha='center', va='center', transform=ax4.transAxes)

# ── 5. Latency vs token count scatter ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
type_colors = {'YN': PALETTE[0], 'Tissue': PALETTE[1], 'Open': PALETTE[2]}
for qtype, grp in rdf.groupby('type'):
    ax5.scatter(grp['tokens'], grp['latency_s'], color=type_colors[qtype],
                label=qtype, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)
ax5.set_title('Latency vs Output Tokens', fontweight='bold')
ax5.set_xlabel('Output Tokens')
ax5.set_ylabel('Latency (s)')
ax5.legend(title='Type')

# ── 6. Per-case heatmap ───────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
heat = rdf.set_index('id')[['score',
                             'has_answer_tag',
                             'has_cot']].copy()
heat.columns = ['score', 'format_ok', 'cot_present']
heat = heat.astype(float).T
sns.heatmap(heat, annot=True, fmt='.2f', ax=ax6, cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, linecolor='#f0f0f0',
            cbar_kws={"label": "Score"})
ax6.set_title('Per-Case Evaluation Heatmap', fontweight='bold')
ax6.tick_params(axis='x', rotation=45)

fig.suptitle('PathOS Distilled (Q8_0) — pathos-gemma4-distilled-GGUF\nEvaluation Results',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('pathos_distilled_eval_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pathos_distilled_eval_results.png')

## 9. Qualitative Review — Raw Outputs

In [ ]:
for _, row in rdf.iterrows():
    sep = '─' * 72
    print(f"\n{sep}")
    print(f"[{row['id']}]  Type: {row['type']}  |  Score: {row['score']}  "
          f"|  CoT words: {row['cot_word_count']}  |  Hedged: {row['hedged']}")
    print(f"Q: {row['question']}")
    expected_str = row.get('expected', row.get('keywords', 'N/A'))
    print(f"Expected: {expected_str}")
    print(f"\n── Response (first 900 chars) ──")
    print(row['raw'][:900])
    if len(row['raw']) > 900:
        print("  ...[truncated]")
    if row['answer']:
        print(f"\n<answer>: {row['answer']}")

## 10. Head-to-Head vs RL Variant (Optional)

If you have also downloaded `pathos-gemma4-distilled-rl-4B-GGUF` (Q4_K_M), run this cell to load it and compare both models side-by-side on a shared subset of questions.

In [ ]:
# ── Optional: load the RL model for comparison ─────────────────────────────
RUN_COMPARISON = False   # set True if you have ~3.4 GB free and the second model

COMPARE_QUESTIONS = [
    {"id": "yn_001",     "question": "A biopsy shows cells with irregular nuclear contours, prominent nucleoli, and a mitotic rate of 8/10 HPF. Is malignancy present?",     "expected": "yes"},
    {"id": "tissue_003", "question": "Skin biopsy shows melanocytes at the dermoepidermal junction with pagetoid upward spread, mitoses, and nuclear pleomorphism. What is the diagnosis?", "keywords": ["melanoma"]},
    {"id": "open_002",   "question": "What IHC markers differentiate pleural mesothelioma from lung adenocarcinoma?", "keywords": ["calretinin", "WT1", "TTF-1", "CEA"]},
]

if RUN_COMPARISON:
    rl_path = hf_hub_download(
        repo_id="dhairyapandya/pathos-gemma4-distilled-rl-4B-GGUF",
        filename="pathos-Q4_K_M.gguf"
    )
    llm_rl = Llama(model_path=rl_path, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)

    def run_with(model, question):
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question}]
        t0 = time.time()
        r  = model.create_chat_completion(messages=msgs, max_tokens=500, temperature=0.1)
        raw = r["choices"][0]["message"]["content"]
        m   = re.search(r'<answer>(.*?)</answer>', raw, re.DOTALL | re.IGNORECASE)
        return raw, m.group(1).strip() if m else None, round(time.time()-t0, 2)

    rows = []
    for q in COMPARE_QUESTIONS:
        raw_sft, ans_sft, lat_sft = run_with(llm,    q["question"])
        raw_rl,  ans_rl,  lat_rl  = run_with(llm_rl, q["question"])
        rows.append({
            "id":          q["id"],
            "question":    q["question"][:80] + "...",
            "SFT answer":  (ans_sft or raw_sft[:120]),
            "SFT latency": lat_sft,
            "RL answer":   (ans_rl  or raw_rl[:120]),
            "RL latency":  lat_rl,
        })

    cmp_df = pd.DataFrame(rows)
    display(cmp_df)
else:
    print("ℹ️  Set RUN_COMPARISON = True to load the RL model and compare.")

## 11. Export Results

In [ ]:
export_cols = [
    "id", "type", "question", "answer",
    "has_answer_tag", "has_cot", "cot_word_count",
    "hedged", "score", "latency_s", "tokens"
]
rdf[export_cols].to_csv("pathos_distilled_eval_results.csv", index=False)

yn_rows     = rdf[rdf["type"] == "YN"]
tissue_rows = rdf[rdf["type"] == "Tissue"]
open_rows   = rdf[rdf["type"] == "Open"]

summary = {
    "model":          "dhairyapandya/pathos-gemma4-distilled-GGUF",
    "quantization":   "Q8_0",
    "training":       "SFT distillation",
    "total_cases":    len(rdf),
    "metrics": {
        "format_compliance":  round(rdf['has_answer_tag'].mean(), 4),
        "anti_hedging_rate":  round((~rdf['hedged']).mean(), 4),
        "cot_presence_rate":  round(rdf['has_cot'].mean(), 4),
        "avg_cot_words":      round(rdf['cot_word_count'].mean(), 1),
        "yn_accuracy":        round(yn_rows['score'].mean(), 4),
        "tissue_accuracy":    round(tissue_rows['score'].mean(), 4),
        "open_kw_coverage":   round(float(open_rows['score'].mean()), 4),
        "avg_latency_s":      round(rdf['latency_s'].mean(), 3),
        "avg_tokens":         round(rdf['tokens'].mean(), 1),
        "avg_tokens_per_sec": round((rdf['tokens'] / rdf['latency_s']).mean(), 1),
    }
}
with open("pathos_distilled_eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Exported:")
print("  pathos_distilled_eval_results.csv")
print("  pathos_distilled_eval_summary.json")
print()
print(json.dumps(summary, indent=2))

## 12. Custom Query Sandbox

In [ ]:
CUSTOM_QUESTION = """
A 70-year-old woman has a 2 cm pancreatic head mass. FNA shows cells with
ductal differentiation, nuclear enlargement, irregular chromatin, and abundant
desmoplastic stroma. What is the most likely diagnosis and what IHC panel
would confirm it?
""".strip()

out = run_inference(CUSTOM_QUESTION, max_tokens=700)

print("=" * 65)
print("QUESTION:\n", CUSTOM_QUESTION)
print("\nMODEL RESPONSE:\n")
print(out['raw'])
print(f"\n<answer>:    {out['answer']}")
print(f"CoT words:   {out['cot_word_count']}")
print(f"Latency:     {out['latency_s']}s")
print(f"Tokens:      {out['tokens']}")
print(f"Hedged:      {out['hedged']}")